In [10]:
pip install google-api-python-client

In [18]:
from googleapiclient.discovery import build
import pandas as pd
from datetime import datetime

import os
API_KEY = os.environ.get('YOUTUBE_API_KEY', '')  # set in env; never commit keys
youtube = build('youtube', 'v3', developerKey=API_KEY)

# 获取频道的订阅者数量
def get_channel_subscribers(channel_id):
    request = youtube.channels().list(
        part="statistics",  # 获取频道统计数据
        id=channel_id  # 传入频道ID
    )
    response = request.execute()
    return response['items'][0]['statistics'].get('subscriberCount', 0)  # 返回订阅者数量

# 获取普通视频（全球或指定国家）
video_data = []
page_token = None

# 设定日期筛选条件
target_date = datetime(2025, 3, 1)  # 目标日期为2025年3月1日

# 请求最多100条数据
for _ in range(2):  # 两次请求每次50条
    request = youtube.search().list(
        part='snippet',  # 获取视频的基本信息
        type='video',  # 获取普通视频
        regionCode='US',  # 设置为美国地区，其他地区可以改成JP/IN/CN等
        maxResults=50,  # 设置获取的最大视频数目
        pageToken=page_token,  # 设置分页token
        publishedAfter='2025-03-01T00:00:00Z',  # 设置发布时间筛选条件
        publishedBefore='2025-03-02T00:00:00Z',  # 设置发布时间筛选条件
    )
    response = request.execute()

    # 获取视频ID列表
    video_ids = [item['id']['videoId'] for item in response['items']]

    # 获取详细视频统计数据
    video_details_request = youtube.videos().list(
        part='snippet,statistics',  # 获取视频的基本信息和统计数据
        id=','.join(video_ids)  # 使用逗号分隔的ID列表
    )
    video_details_response = video_details_request.execute()

    # 处理返回的视频数据
    for item in video_details_response['items']:
        video_id = item['id']  # 获取视频ID
        title = item['snippet']['title']  # 获取视频标题
        channel = item['snippet']['channelTitle']  # 获取频道名称
        channel_id = item['snippet']['channelId']  # 获取频道ID
        published = item['snippet']['publishedAt']  # 获取视频发布时间
        published_date = datetime.strptime(published, '%Y-%m-%dT%H:%M:%SZ')  # 将发布时间转为 datetime 类型
        views = item['statistics'].get('viewCount', 0)  # 获取视频的观看数
        likes = item['statistics'].get('likeCount', 0)  # 获取视频的点赞数
        thumbnail_url = item['snippet']['thumbnails']['high'].get('url', '')  # 获取视频封面图URL
        category_id = item['snippet'].get('categoryId', '')  # 获取视频的类别ID

        # 获取该视频对应频道的订阅者数量
        subscribers = get_channel_subscribers(channel_id)

        # 将每个视频的数据保存到列表中
        video_data.append({
            'video_id': video_id,
            'title': title,
            'channel': channel,
            'channel_id': channel_id,
            'published': published,
            'views': int(views),  # 将观看数转为整数
            'likes': int(likes),  # 将点赞数转为整数
            'thumbnail_url': thumbnail_url,
            'category_id': category_id,
            'subscribers': subscribers  # 添加频道的订阅者数量
        })

    # 设置下一页的token
    page_token = response.get('nextPageToken')

# 将数据保存为CSV文件
df = pd.DataFrame(video_data)  # 将列表转换为DataFrame
df.to_csv('youtube_videos_2025_03_01.csv', index=False)  # 保存为CSV文件，不包含行索引

print("数据已成功保存。")  # 打印保存成功的信息


数据已成功保存。


In [16]:
import requests
import os

def download_thumbnail(url, video_id, save_dir='thumbnails'):
    os.makedirs(save_dir, exist_ok=True)
    response = requests.get(url)
    with open(f"{save_dir}/{video_id}.jpg", 'wb') as f:
        f.write(response.content)

for row in video_data:
    download_thumbnail(row['thumbnail_url'], row['video_id'])

## 将图片导入google drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import shutil

src_folder = 'thumbnails'  # Colab 当前目录里的图片文件夹
dst_folder = '/content/drive/MyDrive/youtube_thumbnails'  # 你云盘里的目标位置

shutil.copytree(src_folder, dst_folder)
print("✅ 已成功上传到 Google Drive！")